# CPPE-5 Object Detection — Notebook thi đấu chuẩn

**Mục tiêu:** xây dựng một pipeline Object Detection có thể dùng như mẫu luyện thi 6 giờ: từ đọc đề → kiểm tra dữ liệu → EDA → chia validation → baseline hợp lệ → train → đánh giá → error analysis → thí nghiệm → chốt model → inference test → tạo submission.

Notebook này dùng **CPPE-5** với 5 lớp PPE:

- `0: Coverall`
- `1: Face_Shield`
- `2: Gloves`
- `3: Goggles`
- `4: Mask`

## Quy tắc giả lập của đề luyện

- Metric chính mặc định: **mAP@0.50**.
- Được dùng **backbone pretrained trên ImageNet**.
- **Không dùng detection checkpoint pretrained trên COCO/OpenImages/Objects365**.
- Không dùng dữ liệu PPE bên ngoài.
- Không dùng nhãn test để chọn model.

> **Quan trọng:** baseline trong notebook dùng `Faster R-CNN + ResNet50-FPN`, với `weights=None` và chỉ nạp `weights_backbone=ImageNet`. Điều này cố ý để phần detection head không nhận trọng số COCO.

## Cách học notebook này

Mỗi code cell đều có:

1. **Trước cell:** mục đích, phải quan sát gì, vì sao bước này tồn tại.
2. **Sau cell:** cách đọc kết quả và nhánh xử lý nếu gặp từng tình huống.

Không nên chỉ `Run all` rồi nhìn điểm. Hãy đọc kết quả từng chặng và ghi lại quyết định.

## Sơ đồ tư duy toàn bài

```text
ĐỌC ĐỀ + RULE
      ↓
XÁC ĐỊNH TASK / METRIC / CONSTRAINT
      ↓
SANITY CHECK DỮ LIỆU
      ↓
EDA DETECTION
      ↓
CHIA TRAIN / VAL VÀ KHÓA SPLIT
      ↓
TẠO DATASET / DATALOADER
      ↓
SMOKE TEST 1 BATCH
      ↓
BASELINE TRAIN
      ↓
VALIDATION mAP + PER-CLASS AP
      ↓
ERROR ANALYSIS
      ↓
ĐẶT GIẢ THUYẾT
      ↓
EXPERIMENT CÓ KIỂM SOÁT
      ↓
CHỐT CONFIG
      ↓
FINAL TRAIN / RETRAIN
      ↓
INFERENCE TEST
      ↓
VALIDATE SUBMISSION
```

### Timeline gợi ý nếu thi 6 giờ

| Mốc | Việc | Deliverable |
|---|---|---|
| 0:00–0:15 | Đọc rule | ghi rõ metric/pretrained/format |
| 0:15–0:45 | Sanity + EDA | biết imbalance, object size |
| 0:45–1:00 | Split | train/val cố định |
| 1:00–1:45 | Baseline | model + metric đầu tiên |
| 1:45–2:15 | Error analysis | biết model sai ở đâu |
| 2:15–4:30 | Experiment | 3–5 thay đổi có lý do |
| 4:30–5:15 | Final | checkpoint tốt nhất |
| 5:15–5:40 | Test inference | submission hợp lệ |
| 5:40–6:00 | Recheck | weights + config + report |

# 1. Cài thư viện

### Mục tiêu

Chỉ cài những gói notebook thực sự cần. Không reinstall `torch/torchvision` nếu môi trường thi đã có sẵn vì dễ gây lệch CUDA.

Ta cần:

- `datasets`: load CPPE-5 trực tiếp.
- `torchmetrics[detection]`: COCO-style mAP.
- `pycocotools`: backend đánh giá detection.

### Khi thi thật

- Nếu BTC cung cấp môi trường cố định, kiểm tra package trước khi cài.
- Nếu Internet bị khóa, ưu tiên thư viện đã có sẵn.
- Không dành 20–30 phút đầu để sửa dependency nếu có thể dùng stack mặc định.

In [ ]:
%pip install -q "datasets>=3.0" "torchmetrics[detection]>=1.4" pycocotools

### Sau cell này — đọc kết quả thế nào?

- **Không có lỗi:** tiếp tục.
- **Lỗi build `pycocotools`:** kiểm tra hệ điều hành/package compiler; trên Colab/Linux thường cài trực tiếp được.
- **Torch/Torchvision xung đột:** không vội reinstall CUDA. In version ra ở cell kế tiếp trước.
- **Thi offline và không cài được:** dùng evaluator BTC cung cấp hoặc chuẩn bị sẵn wheel/package nếu quy chế cho phép.

# 2. Import, cấu hình và seed

### Mục tiêu

Tạo **một nơi duy nhất** chứa các tham số của experiment. Đây là thói quen cực quan trọng trong contest vì giúp:

- so sánh E0/E1/E2 công bằng;
- không quên mình đã đổi tham số nào;
- chạy lại được model;
- giảm lỗi do sửa giá trị rải rác trong notebook.

### Seed

Ta tách:

- `SPLIT_SEED`: chỉ dùng cho train/val split.
- `TRAIN_SEED`: dùng cho random, NumPy, PyTorch, DataLoader.

**Không đổi split giữa các experiment.** Nếu split ban đầu mất cân bằng quá rõ, sửa **trước khi train model đầu tiên**, sau đó khóa lại.

### GPU

- bật AMP để giảm VRAM và tăng tốc;
- bật TF32/cuDNN benchmark khi có CUDA;
- không ép strict deterministic mặc định vì có thể chậm.

In [ ]:
import os
import json
import math
import random
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torchmetrics.detection import MeanAveragePrecision
from tqdm.auto import tqdm

# -------------------------
# CONFIG CHÍNH
# -------------------------
SPLIT_SEED = 2026
TRAIN_SEED = 42
VAL_FRACTION = 0.20

NUM_CLASSES = 5
CLASS_NAMES = ["Coverall", "Face_Shield", "Gloves", "Goggles", "Mask"]

# Metric chính của đề mock: mAP@0.50.
# Nếu đề thật dùng COCO mAP@[.50:.95], đổi thành "map".
PRIMARY_METRIC = "map_50"

ALLOW_IMAGENET_PRETRAIN = True
TRAINABLE_BACKBONE_LAYERS = 3

# Faster R-CNN tự resize ảnh bên trong model.
MIN_SIZE = 640
MAX_SIZE = 1024

BATCH_SIZE = 4
NUM_WORKERS = min(4, os.cpu_count() or 2)
BASELINE_EPOCHS = 8
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
HFLIP_PROB = 0.5
USE_AMP = True
MAX_GRAD_NORM = None

# Giữ threshold thấp khi đánh giá AP để không bỏ prediction quá sớm.
MODEL_SCORE_THRESH = 0.05
MODEL_NMS_THRESH = 0.50
MAX_DETECTIONS_PER_IMAGE = 100

RUN_DIR = Path("cppe5_runs")
RUN_DIR.mkdir(exist_ok=True)
BEST_CKPT = RUN_DIR / "baseline_best.pt"
SPLIT_FILE = RUN_DIR / "split_indices.json"
HISTORY_FILE = RUN_DIR / "baseline_history.csv"

# -------------------------
# REPRODUCIBILITY
# -------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(TRAIN_SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("device     :", DEVICE)

### Sau cell này — phải kiểm tra gì?

**Trường hợp A — `device = cuda`:** tốt, sang kiểm tra GPU.

**Trường hợp B — `device = cpu`:** dừng trước khi train. Nếu đang ở Colab, bật GPU Runtime. Nếu là môi trường BTC, xác minh bạn đang ở đúng node/tài khoản GPU.

**Nếu muốn so sánh experiment:** giữ nguyên tất cả config trừ biến bạn đang kiểm tra. Ví dụ E1 chỉ tăng `MIN_SIZE`, không đồng thời đổi model + augmentation + LR.

**Về seed:** kết quả CUDA vẫn có thể dao động nhẹ dù seed cố định. Mục tiêu contest là **reproducible enough**, không nhất thiết giống từng bit.

# 3. Kiểm tra GPU trước khi đụng vào dữ liệu

### Mục tiêu

Biết chính xác:

- GPU gì;
- VRAM bao nhiêu;
- có đủ để chạy baseline không;
- từ đó chọn batch/image size hợp lý.

Đây là bước giúp tránh mất 20 phút vì OOM ở epoch đầu.

In [ ]:
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024**3
    allocated_gb = torch.cuda.memory_allocated(0) / 1024**3
    reserved_gb = torch.cuda.memory_reserved(0) / 1024**3
    print(f"GPU        : {props.name}")
    print(f"Total VRAM : {total_gb:.2f} GB")
    print(f"Allocated  : {allocated_gb:.2f} GB")
    print(f"Reserved   : {reserved_gb:.2f} GB")
else:
    print("Không có CUDA. Không nên train Faster R-CNN contest baseline bằng CPU.")

### Sau cell này — quyết định batch/image size

Gợi ý an toàn ban đầu cho Faster R-CNN ResNet50-FPN:

- VRAM khoảng **8 GB**: thử batch 2, `MIN_SIZE=512–640`.
- VRAM khoảng **12–16 GB**: thử batch 4, `MIN_SIZE=640`.
- VRAM lớn hơn: có thể thử batch 4–8 hoặc tăng resolution sau khi baseline chạy ổn.

Đây chỉ là điểm bắt đầu, vì số object/ảnh cũng ảnh hưởng memory.

**Nguyên tắc:** đừng dùng 99–100% VRAM. Chừa khoảng 5–15% để tránh spike OOM.

# 4. Load CPPE-5 trực tiếp

### Mục tiêu

Load dữ liệu từ Hugging Face mà không phải tải zip thủ công.

CPPE-5 công khai có:

- 1000 ảnh train;
- 29 ảnh test;
- bbox theo COCO `[x, y, width, height]`;
- 5 class PPE.

Trong bài mock, ta **coi test là private test** và không dùng annotation test để chọn model.

In [ ]:
dataset = load_dataset("rishitdagli/cppe-5")
train_hf = dataset["train"]
test_hf = dataset["test"]

print(dataset)
print("Train images:", len(train_hf))
print("Test images :", len(test_hf))

### Sau cell này — nếu load lỗi thì làm gì?

- **404 / dataset not found:** kiểm tra tên repo chính xác.
- **Connection error:** môi trường không có Internet; trong thi thật phải dùng đường dẫn dataset BTC cung cấp.
- **Dataset load được nhưng schema khác:** tuyệt đối không viết tiếp theo giả định. In sample/schema ở cell kế tiếp trước.

Nếu số lượng khác tài liệu, ưu tiên **dữ liệu thực tế đang nhận được** và đọc README/task statement của cuộc thi.

# 5. Sanity check schema và một sample

### Mục tiêu

Đây là bước chống lỗi nguy hiểm nhất của detection: **hiểu sai bbox hoặc label**.

Ta cần xác nhận:

- `image` là PIL image;
- `objects['bbox']` là `[x, y, w, h]`;
- category là 0–4;
- số bbox = số category;
- bbox có tọa độ hợp lý.

In [ ]:
sample = train_hf[0]

print("Keys:", sample.keys())
print("Image id:", sample["image_id"])
print("Stored width/height:", sample["width"], sample["height"])
print("Decoded image size  :", sample["image"].size)
print("Objects keys:", sample["objects"].keys())
print("Categories:", sample["objects"]["category"])
print("First boxes:", sample["objects"]["bbox"][:5])
print("#boxes == #labels:", len(sample["objects"]["bbox"]) == len(sample["objects"]["category"]))

### Sau cell này — checklist bắt buộc

Bạn phải nói được câu này trước khi tiếp tục:

> “Annotation gốc là COCO `xywh`; model Faster R-CNN sẽ nhận `xyxy`; class gốc 0–4 nhưng torchvision detection dành label 0 cho background, vì vậy khi train ta sẽ dịch class thành 1–5.”

Nếu bbox nhìn như `[x1, y1, x2, y2]` thay vì `[x,y,w,h]`, **dừng và kiểm tra tài liệu**. Sai conversion bbox có thể khiến mAP gần 0 dù training loss vẫn giảm.

# 6. Sanity check toàn bộ annotation

### Mục tiêu

Quét train set để phát hiện:

- bbox width/height <= 0;
- bbox ra ngoài ảnh;
- category sai range;
- ảnh không có object;
- mismatch số box/label.

Nếu dữ liệu có lỗi, phải biết **trước khi model học**.

In [ ]:
issues = []
empty_images = 0

for i in tqdm(range(len(train_hf)), desc="Sanity scan"):
    s = train_hf[i]
    W, H = s["image"].size
    boxes = s["objects"]["bbox"]
    labels = s["objects"]["category"]

    if len(boxes) == 0:
        empty_images += 1

    if len(boxes) != len(labels):
        issues.append((i, "box_label_length_mismatch"))
        continue

    for j, (bbox, cls) in enumerate(zip(boxes, labels)):
        x, y, w, h = map(float, bbox)
        if cls < 0 or cls >= NUM_CLASSES:
            issues.append((i, f"invalid_class_{cls}"))
        if w <= 0 or h <= 0:
            issues.append((i, f"non_positive_box_{j}"))
        if x < -1e-6 or y < -1e-6 or x + w > W + 1 or y + h > H + 1:
            issues.append((i, f"out_of_bounds_box_{j}"))

print("Empty images:", empty_images)
print("Issues found:", len(issues))
print("First issues:", issues[:20])

### Sau cell này — xử lý theo tình huống

**0 issue:** lý tưởng, tiếp tục.

**BBox vượt ảnh rất ít và chỉ lệch 1 pixel:** có thể clamp khi convert, nhưng phải ghi lại.

**Nhiều bbox vượt ảnh:** kiểm tra lại bạn có đọc đúng format không. Đừng “sửa dữ liệu” hàng loạt trước khi hiểu nguyên nhân.

**Có bbox w/h <= 0:** loại annotation lỗi hoặc hỏi BTC nếu là dữ liệu thi.

**Có ảnh 0 object:** Faster R-CNN có thể xử lý target rỗng nếu chuẩn bị đúng, nhưng notebook baseline dưới đây giả định train ảnh có object. Nếu dataset thật có empty image, cần test riêng.

# 7. EDA cấp 1 — số object, class distribution, image resolution

### Mục tiêu

Object detection không thể chỉ nhìn “số ảnh”. Ta cần biết:

- mỗi class có bao nhiêu bbox;
- mỗi class xuất hiện trong bao nhiêu ảnh;
- số object mỗi ảnh;
- resolution ảnh.

Hai khái niệm khác nhau:

- **bbox count:** class xuất hiện bao nhiêu object;
- **image presence:** class xuất hiện trong bao nhiêu ảnh.

Class có nhiều bbox nhưng tập trung vào vài ảnh vẫn có thể gây split không ổn.

In [ ]:
class_box_counts = Counter()
class_image_counts = Counter()
objects_per_image = []
image_rows = []

for i in tqdm(range(len(train_hf)), desc="EDA level 1"):
    s = train_hf[i]
    W, H = s["image"].size
    labels = list(map(int, s["objects"]["category"]))

    objects_per_image.append(len(labels))
    class_box_counts.update(labels)
    class_image_counts.update(set(labels))
    image_rows.append({"idx": i, "width": W, "height": H, "n_objects": len(labels)})

class_df = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "bbox_count": [class_box_counts[i] for i in range(NUM_CLASSES)],
    "image_count": [class_image_counts[i] for i in range(NUM_CLASSES)],
})
class_df["bbox_pct"] = class_df["bbox_count"] / class_df["bbox_count"].sum() * 100

image_df = pd.DataFrame(image_rows)

print(class_df.to_string(index=False))
print("\nObjects/image summary:")
print(pd.Series(objects_per_image).describe(percentiles=[.5, .75, .9, .95, .99]))
print("\nImage resolution summary:")
print(image_df[["width", "height"]].describe())

### Sau cell này — cách đọc EDA

#### Nếu một class có bbox count rất thấp

Ví dụ chỉ bằng 10–20% class lớn nhất:

- đánh dấu đây là **rare class**;
- sau split phải kiểm tra class này đủ mẫu ở validation;
- sau baseline xem `AP_per_class` trước khi nghĩ đến oversampling.

#### Nếu một class có nhiều bbox nhưng ít image

Ví dụ 500 bbox nhưng chỉ trong 40 ảnh:

- random split có thể gây lệch mạnh;
- nên kiểm tra image-level presence ở train/val.

#### Nếu số object/ảnh có p99 rất cao

- memory có thể spike ở batch chứa ảnh dày object;
- đừng ép VRAM sát 100%;
- `MAX_DETECTIONS_PER_IMAGE` phải đủ lớn khi inference.

#### Nếu resolution biến thiên rất lớn

Faster R-CNN tự resize theo `MIN_SIZE/MAX_SIZE`. Sau baseline, nếu object nhỏ yếu, resolution là experiment ưu tiên cao.

# 8. EDA cấp 2 — kích thước và hình dạng bounding box

### Mục tiêu

Đây là EDA quan trọng nhất để quyết định:

- có cần tăng image size không;
- small-object có phải vấn đề chính không;
- aspect ratio có bất thường không;
- class nào đặc biệt nhỏ.

Ta tính `relative_area = bbox_area / image_area` để so sánh giữa ảnh resolution khác nhau.

In [ ]:
bbox_rows = []

for i in tqdm(range(len(train_hf)), desc="BBox EDA"):
    s = train_hf[i]
    W, H = s["image"].size
    for bbox, cls in zip(s["objects"]["bbox"], s["objects"]["category"]):
        x, y, w, h = map(float, bbox)
        bbox_rows.append({
            "image_idx": i,
            "class_id": int(cls),
            "class_name": CLASS_NAMES[int(cls)],
            "w": w,
            "h": h,
            "area": w * h,
            "relative_area": (w * h) / (W * H),
            "aspect_ratio": w / max(h, 1e-9),
        })

bbox_df = pd.DataFrame(bbox_rows)

print("Global bbox statistics:")
print(bbox_df[["w", "h", "area", "relative_area", "aspect_ratio"]].describe(
    percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]
))

print("\nRelative area by class:")
print(
    bbox_df.groupby("class_name")["relative_area"]
    .agg(["count", "median", "mean", "min", "max"])
    .sort_values("median")
)

### Sau cell này — quyết định theo object size

Một quy tắc thực dụng:

- nếu rất nhiều bbox chiếm **< 1% diện tích ảnh**, small-object là nguy cơ lớn;
- nếu class yếu cũng là class có median relative area thấp → ưu tiên thử **resolution cao hơn** trước khi đổi optimizer;
- nếu aspect ratio cực đoan → kiểm tra anchor/default proposals và xem annotation bằng mắt.

### Nhánh hành động

**Small objects nhiều:**

1. baseline 640 trước;
2. nếu AP_small thấp rõ → thử `MIN_SIZE=800` hoặc 960;
3. nếu vẫn thấp → mới cân nhắc custom anchor nhỏ hơn.

**Object chủ yếu lớn:**

- tăng resolution có thể tốn GPU nhưng gain thấp;
- ưu tiên model capacity/augmentation/epoch hơn.

**Một class vừa hiếm vừa nhỏ:** đây thường là class khó nhất; cần nhìn per-class AP + recall sau baseline.

# 9. Visualize ground truth

### Mục tiêu

Không có EDA detection nào hoàn chỉnh nếu chưa nhìn bbox bằng mắt.

Ta xem ảnh ngẫu nhiên để phát hiện:

- annotation sai;
- bbox quá lỏng/quá chặt;
- class dễ nhầm;
- object bị che khuất;
- nhiều object nhỏ.

Trong thi thật, 20–30 ảnh được nhìn kỹ có thể giá trị hơn một experiment mù.

In [ ]:
def show_ground_truth(hf_ds, indices, ncols=3):
    indices = list(indices)
    n = len(indices)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, indices):
        s = hf_ds[int(idx)]
        image = s["image"].convert("RGB")
        ax.imshow(image)
        for bbox, cls in zip(s["objects"]["bbox"], s["objects"]["category"]):
            x, y, w, h = map(float, bbox)
            ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=2))
            ax.text(x, max(0, y-3), CLASS_NAMES[int(cls)], fontsize=8)
        ax.set_title(f"idx={idx} | objects={len(s['objects']['bbox'])}")
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(SPLIT_SEED)
random_indices = rng.choice(len(train_hf), size=9, replace=False)
show_ground_truth(train_hf, random_indices)

### Sau cell này — tự trả lời 5 câu

1. Gloves/Goggles có nhỏ hơn rõ rệt không?
2. Có class nào dễ nhầm bằng mắt không?
3. Bbox có nhất quán không?
4. Có ảnh đông object khiến NMS quan trọng không?
5. Horizontal flip có hợp lý về mặt ngữ nghĩa không?

Nếu annotation nhìn lỗi ở nhiều ảnh, **đừng train ngay**. Xác minh data/parser trước.

# 10. Chia train/validation và khóa split

### Mục tiêu

Validation phải đủ đại diện để ta tin vào quyết định experiment.

Baseline dùng split 80/20 với `SPLIT_SEED=2026`. Sau đó ta sẽ **kiểm tra phân bố**. Nếu split quá lệch, có thể đổi seed **một lần dựa trên phân bố dữ liệu**, không dựa trên mAP, rồi khóa lại.

### Không được làm

- E0 dùng seed 42, E1 dùng 123, E2 dùng 999.
- thử 20 split rồi chọn split nào cho mAP cao nhất.

Đó là overfit validation.

In [ ]:
all_indices = np.arange(len(train_hf))
train_idx, val_idx = train_test_split(
    all_indices,
    test_size=VAL_FRACTION,
    random_state=SPLIT_SEED,
    shuffle=True,
)

split_payload = {
    "split_seed": SPLIT_SEED,
    "val_fraction": VAL_FRACTION,
    "train_idx": train_idx.tolist(),
    "val_idx": val_idx.tolist(),
}
SPLIT_FILE.write_text(json.dumps(split_payload, indent=2), encoding="utf-8")

print("Train images:", len(train_idx))
print("Val images  :", len(val_idx))
print("Saved split:", SPLIT_FILE)

### Sau cell này

Từ thời điểm này, coi `split_indices.json` là **đóng băng**.

Chỉ thay split nếu cell chẩn đoán tiếp theo cho thấy một class gần như biến mất khỏi validation hoặc lệch quá lớn. Nếu sửa, sửa trước E0 rồi lưu lại file split mới.

# 11. Kiểm tra chất lượng split

### Mục tiêu

So sánh **image presence** và **bbox count** theo class giữa train và val.

Ta không cần tỷ lệ giống tuyệt đối, nhưng không muốn class hiếm chỉ có vài object ở validation vì AP khi đó rất nhiễu.

In [ ]:
def split_class_report(hf_ds, indices, split_name):
    box_counts = Counter()
    image_counts = Counter()
    for idx in indices:
        labels = list(map(int, hf_ds[int(idx)]["objects"]["category"]))
        box_counts.update(labels)
        image_counts.update(set(labels))
    return pd.DataFrame({
        "split": split_name,
        "class_id": range(NUM_CLASSES),
        "class_name": CLASS_NAMES,
        "bbox_count": [box_counts[i] for i in range(NUM_CLASSES)],
        "image_count": [image_counts[i] for i in range(NUM_CLASSES)],
    })

split_report = pd.concat([
    split_class_report(train_hf, train_idx, "train"),
    split_class_report(train_hf, val_idx, "val"),
], ignore_index=True)

pivot_boxes = split_report.pivot(index="class_name", columns="split", values="bbox_count")
pivot_images = split_report.pivot(index="class_name", columns="split", values="image_count")

print("BBox counts:")
print(pivot_boxes)
print("\nImage presence:")
print(pivot_images)

### Sau cell này — khi nào split được coi là “đủ tốt”?

Không có ngưỡng thần kỳ, nhưng với dataset 1000 ảnh:

- mọi class nên xuất hiện trong validation với số lượng **đủ để metric không quá nhiễu**;
- tỷ lệ train/val theo class không nên lệch cực đoan;
- class hiếm phải đặc biệt được kiểm tra.

### Nếu split xấu

- đổi `SPLIT_SEED` sang một seed khác **dựa trên phân bố này**, không train thử rồi chọn bằng mAP;
- rerun từ config → split → report;
- khi thấy hợp lý, **khóa seed vĩnh viễn cho toàn bộ experiment**.

### Nếu split tốt

Không chạm vào `SPLIT_SEED` nữa.

# 12. Dataset adapter cho Torchvision Detection

### Mục tiêu

Chuyển CPPE-5 sang format Faster R-CNN cần:

- image tensor `[C,H,W]` trong `[0,1]`;
- boxes dạng `xyxy` tuyệt đối;
- labels dạng `1..5` vì `0` được dành cho background.

### Augmentation baseline

Chỉ dùng **horizontal flip** xác suất 0.5. Đây là augmentation dễ hiểu và ít rủi ro.

Không dùng vertical flip mặc định vì ảnh người/PPE lộn ngược không tự nhiên.

In [ ]:
class CPPEDetectionDataset(Dataset):
    def __init__(self, hf_dataset, indices, train=False, hflip_prob=0.0):
        self.hf_dataset = hf_dataset
        self.indices = np.asarray(indices)
        self.train = train
        self.hflip_prob = float(hflip_prob)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos):
        idx = int(self.indices[pos])
        s = self.hf_dataset[idx]
        image = s["image"].convert("RGB")
        W, H = image.size

        boxes_xywh = torch.as_tensor(s["objects"]["bbox"], dtype=torch.float32)
        labels = torch.as_tensor(s["objects"]["category"], dtype=torch.int64) + 1

        if boxes_xywh.numel() == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
        else:
            boxes = boxes_xywh.clone()
            boxes[:, 2] = boxes_xywh[:, 0] + boxes_xywh[:, 2]  # x2 = x + w
            boxes[:, 3] = boxes_xywh[:, 1] + boxes_xywh[:, 3]  # y2 = y + h

            # Clamp nhẹ để chống sai số biên 1 pixel.
            boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, W)
            boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, H)

            keep = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
            boxes = boxes[keep]
            labels = labels[keep]

        if self.train and self.hflip_prob > 0 and random.random() < self.hflip_prob:
            image = TF.hflip(image)
            if len(boxes) > 0:
                x1 = boxes[:, 0].clone()
                x2 = boxes[:, 2].clone()
                boxes[:, 0] = W - x2
                boxes[:, 2] = W - x1

        image = TF.pil_to_tensor(image).float() / 255.0

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(int(s["image_id"]), dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(TRAIN_SEED)

train_ds = CPPEDetectionDataset(train_hf, train_idx, train=True, hflip_prob=HFLIP_PROB)
val_ds = CPPEDetectionDataset(train_hf, val_idx, train=False, hflip_prob=0.0)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn,
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

### Sau cell này — lỗi thường gặp

**`labels` vẫn là 0–4:** sai với torchvision Faster R-CNN training; phải là 1–5.

**BBox sau flip bị âm:** công thức flip sai hoặc lấy width sai.

**DataLoader treo:** giảm `NUM_WORKERS` về 2 hoặc 0 để chẩn đoán.

**GPU chờ dữ liệu lâu:** sau khi pipeline đúng, tăng worker/pin memory/cache mới là bước tối ưu.

# 13. Kiểm tra một batch sau conversion

### Mục tiêu

Trước khi tạo model, xác nhận:

- tensor ảnh đúng range;
- box là xyxy hợp lệ;
- label đúng 1–5;
- không có box âm/zero area.

Đây là “unit test” rẻ hơn rất nhiều so với train một epoch rồi mới phát hiện format sai.

In [ ]:
images, targets = next(iter(train_loader))

print("Batch size:", len(images))
print("Image tensor shape:", images[0].shape)
print("Image range:", float(images[0].min()), float(images[0].max()))
print("Boxes shape:", targets[0]["boxes"].shape)
print("Labels unique:", torch.unique(targets[0]["labels"]))
print("First boxes xyxy:\n", targets[0]["boxes"][:5])

for t in targets:
    if len(t["boxes"]) > 0:
        assert torch.all(t["boxes"][:, 2] > t["boxes"][:, 0])
        assert torch.all(t["boxes"][:, 3] > t["boxes"][:, 1])
        assert torch.all((t["labels"] >= 1) & (t["labels"] <= NUM_CLASSES))

print("Batch sanity: OK")

### Sau cell này

Nếu `Batch sanity: OK`, parser/adapter cơ bản đã an toàn.

Nếu fail:

- đừng “fix trong training loop”;
- quay lại adapter;
- visualize chính sample lỗi để hiểu conversion.

Trong contest, lỗi data adapter là loại lỗi nên xử lý dứt điểm trước khi train.

# 14. Tạo baseline model hợp lệ với rule mock

### Vì sao chọn Faster R-CNN + ResNet50-FPN?

- baseline detection kinh điển, dễ debug;
- FPN hữu ích khi object có nhiều kích thước;
- torchvision có pipeline training/inference hoàn chỉnh;
- cho phép **ImageNet-pretrained backbone nhưng detection head random**.

### Điểm cực quan trọng về pretrained

Ta dùng:

- `weights=None` → **không** load Faster R-CNN COCO weights;
- `weights_backbone=ResNet50_Weights.IMAGENET1K_V1` → chỉ backbone được pretrained ImageNet.

Nếu đề thật cấm mọi pretrained, đặt `ALLOW_IMAGENET_PRETRAIN=False`.

In [ ]:
def build_model():
    backbone_weights = (
        ResNet50_Weights.IMAGENET1K_V1
        if ALLOW_IMAGENET_PRETRAIN
        else None
    )

    model = fasterrcnn_resnet50_fpn(
        weights=None,  # QUAN TRỌNG: không dùng COCO detector weights
        weights_backbone=backbone_weights,
        num_classes=NUM_CLASSES + 1,  # + background
        trainable_backbone_layers=(
            TRAINABLE_BACKBONE_LAYERS if ALLOW_IMAGENET_PRETRAIN else 5
        ),
        min_size=MIN_SIZE,
        max_size=MAX_SIZE,
        box_score_thresh=MODEL_SCORE_THRESH,
        box_nms_thresh=MODEL_NMS_THRESH,
        box_detections_per_img=MAX_DETECTIONS_PER_IMAGE,
    )
    return model

model = build_model().to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M")

### Sau cell này — kiểm tra tính hợp lệ trước khi train

Nếu quy chế chỉ cho ImageNet backbone, cấu hình trên hợp lệ về ý tưởng.

**Không đổi thành `weights='DEFAULT'` hoặc COCO weights** nếu rule không cho phép, vì torchvision `DEFAULT` cho Faster R-CNN tương ứng detection weights pretrained COCO.

Nếu GPU yếu, model đầu tiên để giảm chi phí thường là:

1. giảm `BATCH_SIZE`;
2. giảm `MIN_SIZE`;
3. sau đó mới cân nhắc model nhẹ hơn.

# 15. Smoke test 1 batch trước khi train thật

### Mục tiêu

Chạy **một forward training** để phát hiện ngay:

- OOM;
- label/box format sai;
- NaN loss;
- model/data không tương thích.

Nếu smoke test không qua, tuyệt đối chưa chạy epoch dài.

In [ ]:
model.train()
smoke_images, smoke_targets = next(iter(train_loader))
smoke_images = [img.to(DEVICE, non_blocking=True) for img in smoke_images]
smoke_targets = [
    {k: v.to(DEVICE, non_blocking=True) for k, v in t.items()}
    for t in smoke_targets
]

amp_enabled = USE_AMP and DEVICE.type == "cuda"

with torch.cuda.amp.autocast(enabled=amp_enabled):
    loss_dict = model(smoke_images, smoke_targets)
    total_loss = sum(loss_dict.values())

print({k: float(v.detach().cpu()) for k, v in loss_dict.items()})
print("Total loss:", float(total_loss.detach().cpu()))
print("Finite:", bool(torch.isfinite(total_loss)))

# Giải phóng graph smoke test trước training chính.
del smoke_images, smoke_targets, loss_dict, total_loss
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### Sau cell này — nhánh xử lý

**OOM:**

1. giảm batch 4 → 2;
2. giữ AMP bật;
3. nếu vẫn OOM, giảm `MIN_SIZE` 640 → 512;
4. không tắt validation hoặc sửa random để “né” OOM.

**Loss = NaN/Inf:**

- kiểm tra box zero/negative;
- LR chưa tham gia ở smoke forward nên thường là dữ liệu/model input;
- thử một batch khác để xác định sample lỗi.

**Loss finite:** baseline đủ điều kiện để train.

# 16. Hàm đánh giá COCO mAP

### Mục tiêu

Theo dõi cả:

- `map`: COCO mAP@[.50:.95];
- `map_50`: AP@IoU=0.50;
- `map_75`: localization khó hơn;
- `map_small/medium/large`;
- `map_per_class`.

Dù đề mock chấm `map_50`, các metric phụ giúp **chẩn đoán model**.

Ví dụ:

- AP50 cao nhưng AP75 thấp → localization chưa khít;
- AP_small thấp → vấn đề small object;
- một class AP thấp → class-specific issue, không nên đổi cả pipeline mù quáng.

In [ ]:
@torch.inference_mode()
def evaluate_map(model, data_loader, device):
    model.eval()
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        class_metrics=True,
    )

    for images, targets in tqdm(data_loader, desc="Validation", leave=False):
        images = [img.to(device, non_blocking=True) for img in images]
        outputs = model(images)

        preds_cpu = []
        targets_cpu = []

        for out, tgt in zip(outputs, targets):
            preds_cpu.append({
                "boxes": out["boxes"].detach().cpu(),
                "scores": out["scores"].detach().cpu(),
                "labels": out["labels"].detach().cpu(),
            })
            targets_cpu.append({
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu(),
                "area": tgt["area"].detach().cpu(),
                "iscrowd": tgt["iscrowd"].detach().cpu(),
            })

        metric.update(preds_cpu, targets_cpu)

    return metric.compute()


def scalar_metric_dict(result):
    keys = ["map", "map_50", "map_75", "map_small", "map_medium", "map_large",
            "mar_1", "mar_10", "mar_100"]
    out = {}
    for k in keys:
        v = result.get(k)
        if v is not None and torch.is_tensor(v) and v.numel() == 1:
            out[k] = float(v.cpu())
    return out

### Sau cell này

Đây mới là evaluator; chưa có điểm cho tới khi model được train.

**Lưu ý metric:** `map` và `map_50` không thay thế nhau. Khi đề chấm `map_50`, dùng `PRIMARY_METRIC='map_50'` để chọn checkpoint, nhưng vẫn xem `map_75` và size metrics để chẩn đoán.

# 17. Training loop baseline

### Mục tiêu

Training loop cần:

- AMP;
- optimizer rõ ràng;
- kiểm tra finite loss;
- log từng loại loss;
- scheduler;
- save best checkpoint theo metric chính.

### Vì sao SGD?

SGD + momentum là baseline lâu đời và ổn cho Faster R-CNN. Trong contest, baseline nên **dễ giải thích và ít biến số** trước khi thử optimizer khác.

In [ ]:
def train_one_epoch(model, data_loader, optimizer, scaler, device):
    model.train()
    running = defaultdict(float)
    n_batches = 0

    pbar = tqdm(data_loader, desc="Train", leave=False)
    for images, targets in pbar:
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [
            {k: v.to(device, non_blocking=True) for k, v in t.items()}
            for t in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss detected: {float(loss.detach().cpu())}")

        scaler.scale(loss).backward()

        if MAX_GRAD_NORM is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        scaler.step(optimizer)
        scaler.update()

        running["loss_total"] += float(loss.detach().cpu())
        for k, v in loss_dict.items():
            running[k] += float(v.detach().cpu())
        n_batches += 1

        pbar.set_postfix(loss=f"{running['loss_total']/n_batches:.4f}")

    return {k: v / max(n_batches, 1) for k, v in running.items()}


def run_baseline_training(model):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params,
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    milestones = sorted(set([max(1, BASELINE_EPOCHS // 2), max(2, int(BASELINE_EPOCHS * 0.75))]))
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=milestones,
        gamma=0.1,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == "cuda"))

    history = []
    best_score = -float("inf")
    best_epoch = -1

    for epoch in range(1, BASELINE_EPOCHS + 1):
        start = time.time()
        train_stats = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE)
        val_result = evaluate_map(model, val_loader, DEVICE)
        val_scalars = scalar_metric_dict(val_result)
        score = val_scalars[PRIMARY_METRIC]
        elapsed = time.time() - start

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": elapsed,
            **train_stats,
            **val_scalars,
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{BASELINE_EPOCHS} | "
            f"loss={row['loss_total']:.4f} | "
            f"mAP={row.get('map', float('nan')):.4f} | "
            f"mAP50={row.get('map_50', float('nan')):.4f} | "
            f"mAP75={row.get('map_75', float('nan')):.4f} | "
            f"time={elapsed/60:.1f}m"
        )

        if score > best_score:
            best_score = score
            best_epoch = epoch
            torch.save({
                "model_state": model.state_dict(),
                "epoch": epoch,
                "best_score": best_score,
                "primary_metric": PRIMARY_METRIC,
                "config": {
                    "split_seed": SPLIT_SEED,
                    "train_seed": TRAIN_SEED,
                    "min_size": MIN_SIZE,
                    "max_size": MAX_SIZE,
                    "batch_size": BATCH_SIZE,
                    "lr": LR,
                    "epochs": BASELINE_EPOCHS,
                    "hflip_prob": HFLIP_PROB,
                    "imagenet_pretrain": ALLOW_IMAGENET_PRETRAIN,
                },
            }, BEST_CKPT)

        scheduler.step()
        pd.DataFrame(history).to_csv(HISTORY_FILE, index=False)

    print(f"Best {PRIMARY_METRIC}: {best_score:.4f} at epoch {best_epoch}")
    print("Checkpoint:", BEST_CKPT)
    return pd.DataFrame(history)

### Sau cell này

Hàm đã sẵn sàng nhưng chưa train. Cell kế tiếp mới tốn GPU thật.

Trước khi chạy, đảm bảo:

- smoke test đã OK;
- split đã khóa;
- GPU còn đủ VRAM;
- bạn ghi lại baseline config.

# 18. Train baseline E0

### Mục tiêu

Tạo **điểm neo E0**. Mọi experiment sau phải trả lời:

> “Tôi thay đổi gì so với E0, vì sao, và metric nào cải thiện?”

Không đánh giá baseline bằng cảm giác. Lưu history + checkpoint.

In [ ]:
history_df = run_baseline_training(model)
history_df.tail()

### Sau cell này — cách đọc training

#### Case 1 — loss giảm, mAP tăng đều tới epoch cuối

Model chưa hội tụ. Experiment ưu tiên:

- tăng epoch;
- giữ kiến trúc và augmentation trước;
- đừng đổi 5 thứ cùng lúc.

#### Case 2 — loss tiếp tục giảm nhưng val mAP đạt đỉnh rồi giảm

Dấu hiệu overfit:

- dùng best checkpoint thay vì epoch cuối;
- tăng augmentation hợp lý;
- tăng weight decay hoặc freeze backbone nhiều hơn;
- không đơn giản train lâu hơn.

#### Case 3 — loss cao và mAP thấp từ đầu tới cuối

Khả năng underfit hoặc config sai:

- xem prediction bằng mắt;
- kiểm tra LR;
- tăng trainable backbone layers / epoch;
- xác minh bbox/label lại trước khi đổi model.

#### Case 4 — mAP gần 0 dù loss giảm

Ưu tiên nghi ngờ **pipeline/metric/label conversion**, không phải “model yếu”.

# 19. Vẽ lịch sử baseline

### Mục tiêu

Đồ thị giúp phân biệt:

- chưa hội tụ;
- overfit;
- metric dao động mạnh;
- scheduler giảm LR quá sớm.

Ta vẽ riêng loss và metric, không dùng subplot để dễ đọc.

In [ ]:
if HISTORY_FILE.exists():
    history_df = pd.read_csv(HISTORY_FILE)

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["loss_total"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Baseline training loss")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["map_50"], marker="o", label="mAP@0.50")
plt.plot(history_df["epoch"], history_df["map"], marker="o", label="mAP@[.50:.95]")
plt.plot(history_df["epoch"], history_df["map_75"], marker="o", label="mAP@0.75")
plt.xlabel("Epoch")
plt.ylabel("AP")
plt.title("Validation metrics")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Sau cell này — câu hỏi quyết định

- Best metric nằm ở epoch nào?
- 2 epoch cuối còn tăng hay đã plateau?
- AP50 và AP75 cách nhau lớn không?
- Nếu gap AP50–AP75 lớn, vấn đề có thể là localization hơn là classification.

Không tăng epoch chỉ vì “8 epoch nghe ít”. Hãy nhìn curve.

# 20. Load best checkpoint và đánh giá chi tiết

### Mục tiêu

Không dùng model của epoch cuối nếu best checkpoint ở epoch trước.

Sau đó lấy:

- mAP tổng;
- AP theo size;
- AP theo class.

Đây là cell quan trọng nhất để quyết định experiment tiếp theo.

In [ ]:
best_payload = torch.load(BEST_CKPT, map_location=DEVICE)
best_model = build_model().to(DEVICE)
best_model.load_state_dict(best_payload["model_state"])

best_result = evaluate_map(best_model, val_loader, DEVICE)
print("Best epoch:", best_payload["epoch"])
print("Primary metric:", best_payload["primary_metric"], best_payload["best_score"])
print("\nGlobal metrics:")
for k, v in scalar_metric_dict(best_result).items():
    print(f"{k:12s}: {v:.4f}")

classes = best_result["classes"].cpu().tolist()
map_per_class = best_result["map_per_class"].cpu().tolist()
mar_per_class = best_result["mar_100_per_class"].cpu().tolist()

rows = []
for label, ap, ar in zip(classes, map_per_class, mar_per_class):
    # label của model là 1..5
    class_idx = int(label) - 1
    rows.append({
        "model_label": int(label),
        "class_name": CLASS_NAMES[class_idx] if 0 <= class_idx < NUM_CLASSES else str(label),
        "mAP_50_95_per_class": float(ap),
        "mAR100_per_class": float(ar),
    })

per_class_df = pd.DataFrame(rows).sort_values("mAP_50_95_per_class")
print("\nPer-class metrics:")
print(per_class_df.to_string(index=False))

### Sau cell này — decision tree quan trọng

#### A. `map_small << map_medium/map_large`

**Vấn đề chính:** small object.

Thử theo thứ tự:

1. tăng `MIN_SIZE` 640 → 800;
2. kiểm tra GPU/time;
3. nếu gain tốt mới thử 960;
4. sau đó mới cân nhắc custom anchors nhỏ.

#### B. Một class AP thấp rõ nhưng size không nhỏ hơn

Kiểm tra:

- bbox count/image count;
- class có dễ nhầm bằng mắt không;
- annotation có nhất quán không;
- recall thấp hay precision thấp.

#### C. AP50 tốt nhưng AP75 rất thấp

Model tìm đúng object nhưng bbox chưa chính xác:

- resolution;
- train lâu hơn nếu curve còn tăng;
- augmentation quá mạnh có thể làm localization khó;
- kiểm tra label quality.

#### D. Tất cả class đều thấp tương tự

Đây là vấn đề toàn cục:

- underfit;
- resolution quá thấp;
- LR/epoch;
- backbone chưa fine-tune đủ;
- data pipeline.

# 21. Visualize prediction của best model

### Mục tiêu

Metric nói “bao nhiêu”; visualization nói **“sai kiểu gì”**.

Ta overlay:

- GT bằng đường liền;
- prediction bằng đường nét đứt;
- chỉ hiển thị prediction trên ngưỡng để hình không quá rối.

Ngưỡng visualization không phải ngưỡng tối ưu AP.

In [ ]:
@torch.inference_mode()
def show_predictions(model, dataset_obj, positions, score_threshold=0.30, ncols=2):
    model.eval()
    positions = list(positions)
    n = len(positions)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 6*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, pos in zip(axes, positions):
        image, target = dataset_obj[int(pos)]
        output = model([image.to(DEVICE)])[0]

        ax.imshow(image.permute(1, 2, 0).cpu())

        # GT: đường liền
        for box, label in zip(target["boxes"], target["labels"]):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2, linestyle="-"))
            ax.text(x1, y1, f"GT:{CLASS_NAMES[int(label)-1]}", fontsize=8)

        # Prediction: đường nét đứt
        keep = output["scores"] >= score_threshold
        for box, label, score in zip(output["boxes"][keep], output["labels"][keep], output["scores"][keep]):
            x1, y1, x2, y2 = box.detach().cpu().tolist()
            cls = int(label.detach().cpu()) - 1
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2, linestyle="--"))
            ax.text(x1, max(0, y1+12), f"P:{CLASS_NAMES[cls]} {float(score):.2f}", fontsize=8)

        ax.set_title(f"val position={pos}")
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(123)
positions = rng.choice(len(val_ds), size=min(6, len(val_ds)), replace=False)
show_predictions(best_model, val_ds, positions, score_threshold=0.30)

### Sau cell này — phân loại lỗi bằng mắt

Ghi mỗi lỗi vào một trong 4 nhóm:

1. **Missed detection (FN):** GT có, model bỏ sót.
2. **False positive (FP):** model phát hiện vật không tồn tại.
3. **Classification error:** box khá đúng nhưng class sai.
4. **Localization error:** class đúng nhưng box lệch/không khít.

### Hành động theo lỗi

**FN nhiều ở object nhỏ:** resolution/anchor/recall.

**FP nhiều:** hard negative, confidence/NMS, augmentation, class confusion.

**Class sai:** inspect class similarity + imbalance.

**Box lệch:** AP75 thấp; ưu tiên localization, không phải threshold class.

# 22. Diagnostic TP/FP/FN theo class ở một confidence threshold

### Mục tiêu

AP là metric xếp hạng qua nhiều threshold, rất tốt để chấm thi nhưng đôi khi khó chẩn đoán trực tiếp.

Cell này tạo một **diagnostic operating point** tại confidence threshold cố định để xem class nào:

- precision thấp;
- recall thấp;
- F1 thấp.

Đây **không thay thế mAP**.

In [ ]:
@torch.inference_mode()
def diagnostic_tp_fp_fn(model, data_loader, device, score_threshold=0.30, iou_threshold=0.50):
    model.eval()
    stats = {c: {"tp": 0, "fp": 0, "fn": 0} for c in range(1, NUM_CLASSES + 1)}

    for images, targets in tqdm(data_loader, desc="Diagnostic", leave=False):
        images_dev = [img.to(device, non_blocking=True) for img in images]
        outputs = model(images_dev)

        for out, tgt in zip(outputs, targets):
            pboxes = out["boxes"].detach().cpu()
            plabels = out["labels"].detach().cpu()
            pscores = out["scores"].detach().cpu()
            gboxes = tgt["boxes"].cpu()
            glabels = tgt["labels"].cpu()

            keep = pscores >= score_threshold
            pboxes, plabels, pscores = pboxes[keep], plabels[keep], pscores[keep]

            for cls in range(1, NUM_CLASSES + 1):
                pb = pboxes[plabels == cls]
                ps = pscores[plabels == cls]
                gb = gboxes[glabels == cls]

                if len(pb) == 0:
                    stats[cls]["fn"] += len(gb)
                    continue
                if len(gb) == 0:
                    stats[cls]["fp"] += len(pb)
                    continue

                order = torch.argsort(ps, descending=True)
                pb = pb[order]
                ious = box_iou(pb, gb)
                matched_gt = set()

                for pi in range(len(pb)):
                    best_iou, best_gi = torch.max(ious[pi], dim=0)
                    gi = int(best_gi)
                    if float(best_iou) >= iou_threshold and gi not in matched_gt:
                        stats[cls]["tp"] += 1
                        matched_gt.add(gi)
                    else:
                        stats[cls]["fp"] += 1

                stats[cls]["fn"] += len(gb) - len(matched_gt)

    rows = []
    for cls in range(1, NUM_CLASSES + 1):
        tp, fp, fn = stats[cls]["tp"], stats[cls]["fp"], stats[cls]["fn"]
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        rows.append({
            "class_name": CLASS_NAMES[cls-1],
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision@thr": precision,
            "recall@thr": recall,
            "f1@thr": f1,
        })
    return pd.DataFrame(rows).sort_values("f1@thr")

operating_df = diagnostic_tp_fp_fn(
    best_model,
    val_loader,
    DEVICE,
    score_threshold=0.30,
    iou_threshold=0.50,
)
print(operating_df.to_string(index=False))

### Sau cell này — đọc precision/recall thế nào?

#### Precision cao, Recall thấp

Model “thận trọng”, ít FP nhưng bỏ sót nhiều.

Nếu class đó cũng có nhiều small object:

- tăng resolution;
- xem RPN/anchor;
- train thêm nếu curve chưa hội tụ.

Đừng chỉ hạ threshold rồi kết luận model tốt hơn — mAP phải xác nhận.

#### Recall cao, Precision thấp

Model bắt nhiều nhưng FP lớn:

- inspect background/hard negative;
- NMS/confidence có thể giúp operating point;
- augmentation hoặc class confusion có thể là nguyên nhân học thuật.

#### Precision và Recall đều thấp

- feature/model chưa học class;
- class hiếm;
- annotation khó;
- model capacity/resolution.

Luôn đối chiếu với `per_class_df` và ảnh prediction.

# 23. Bảng quyết định experiment — đừng tuning mù

Dùng bảng này sau baseline:

| Triệu chứng | Khả năng | Experiment ưu tiên | Vì sao |
|---|---|---|---|
| AP_small thấp rõ | object nhỏ | `MIN_SIZE 640→800` | nhiều pixel hơn cho object |
| AP50 cao, AP75 thấp | localization | resolution / train thêm | box chưa khít |
| mAP còn tăng ở epoch cuối | chưa hội tụ | tăng epoch | model chưa học xong |
| train loss giảm, val mAP giảm | overfit | augmentation / freeze hơn / WD | giảm variance |
| 1 class rare + recall thấp | thiếu tín hiệu | sampling/augmentation class-aware | tăng exposure |
| mọi class cùng thấp | underfit/config | LR/epoch/backbone/model | vấn đề toàn cục |
| GPU util thấp | input bottleneck | workers/pin/batch | GPU đang chờ CPU |
| OOM | memory | batch↓ trước, size↓ sau | giữ signal ảnh nếu có thể |
| mAP≈0 | pipeline bug | bbox/label/metric check | model yếu hiếm khi về đúng 0 |

### Thứ tự experiment mình khuyên

**E0** Baseline hiện tại.

**E1** Nếu small-object yếu: tăng resolution.

**E2** Nếu curve chưa hội tụ: tăng epoch.

**E3** Nếu underfit: fine-tune thêm backbone layer / capacity.

**E4** Nếu overfit: augmentation hợp lý.

**E5** Chỉ khi có bằng chứng: custom anchors / sampling / optimizer khác.

> Mỗi experiment chỉ nên thay **một nhóm giả thuyết chính**.

# 24. Experiment log

### Mục tiêu

Không dựa vào trí nhớ. Mỗi run ghi:

- thay đổi gì;
- best epoch;
- mAP/mAP50/mAP75;
- nhận xét ngắn;
- thời gian.

Cell này tạo/đọc file log để bạn cập nhật sau mỗi experiment.

In [ ]:
EXPERIMENT_LOG = RUN_DIR / "experiment_log.csv"

columns = [
    "exp", "change", "min_size", "batch_size", "epochs", "lr",
    "map", "map_50", "map_75", "map_small", "map_medium", "map_large",
    "best_epoch", "notes"
]

if EXPERIMENT_LOG.exists():
    exp_df = pd.read_csv(EXPERIMENT_LOG)
else:
    exp_df = pd.DataFrame(columns=columns)

baseline_summary = scalar_metric_dict(best_result)
baseline_row = {
    "exp": "E0",
    "change": "FasterRCNN R50-FPN, ImageNet backbone, hflip baseline",
    "min_size": MIN_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": BASELINE_EPOCHS,
    "lr": LR,
    "map": baseline_summary.get("map"),
    "map_50": baseline_summary.get("map_50"),
    "map_75": baseline_summary.get("map_75"),
    "map_small": baseline_summary.get("map_small"),
    "map_medium": baseline_summary.get("map_medium"),
    "map_large": baseline_summary.get("map_large"),
    "best_epoch": best_payload["epoch"],
    "notes": "Baseline",
}

if not ((exp_df["exp"] == "E0").any() if len(exp_df) else False):
    exp_df = pd.concat([exp_df, pd.DataFrame([baseline_row])], ignore_index=True)
    exp_df.to_csv(EXPERIMENT_LOG, index=False)

print(exp_df.to_string(index=False))
print("Log file:", EXPERIMENT_LOG)

### Sau cell này

Khi chạy E1/E2/E3, thêm một row mới thay vì overwrite E0.

Nếu hai experiment chênh rất nhỏ, ví dụ 0.501 vs 0.503:

- chưa chắc gain thật;
- nếu còn thời gian, rerun top-2 config bằng seed training khác;
- **không đổi validation split**.

Nếu chênh lớn và nhất quán ở per-class/visualization, ưu tiên tiến lên thay vì tốn giờ chạy 5 seed.

# 25. Chốt model trước khi test

Trước khi inference test, phải “freeze decision”:

- config đã chọn;
- checkpoint đã chọn;
- không nhìn test annotation;
- không chỉnh model bằng cảm giác từ test.

### Có nên retrain trên toàn bộ 1000 ảnh?

**Có thể**, sau khi hyperparameter đã khóa.

Cách an toàn:

- lấy `best_epoch` từ validation;
- build model mới từ ImageNet backbone;
- train trên toàn bộ labeled train với số epoch cố định gần `best_epoch`;
- không dùng test để early stop.

Nhưng trong contest 6 giờ, retrain full chỉ nên làm khi còn đủ thời gian. Nếu không, dùng best validation checkpoint vẫn tốt hơn một final run chưa xong.

# 26. Optional: chuẩn bị full-train loader để final retrain

### Mục tiêu

Cell này **không tự retrain**. Nó chỉ chuẩn bị DataLoader toàn bộ 1000 ảnh để bạn dùng khi đã chốt recipe.

Nếu thời gian thi còn ít, bỏ qua final retrain và dùng `best_model`.

In [ ]:
full_train_indices = np.arange(len(train_hf))
full_train_ds = CPPEDetectionDataset(
    train_hf,
    full_train_indices,
    train=True,
    hflip_prob=HFLIP_PROB,
)

full_train_loader = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

print("Full-train images:", len(full_train_ds))
print("Suggested fixed final epochs from baseline best:", best_payload["epoch"])

### Sau cell này

**Không train full chỉ vì “có vẻ chuyên nghiệp”.**

Final retrain đáng làm khi:

- config đã ổn;
- best epoch rõ;
- còn đủ thời gian;
- pipeline inference/submission đã test được.

Nếu chưa từng tạo submission thử, ưu tiên submission pipeline trước final retrain.

# 27. Tạo test dataset không dùng annotation

### Mục tiêu

Test loader chỉ cần image và image_id. Ta cố tình không đọc `objects` để mô phỏng private test đúng nghĩa.

In [ ]:
class CPPETestDataset(Dataset):
    def __init__(self, hf_dataset):
        self.hf_dataset = hf_dataset

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        s = self.hf_dataset[int(idx)]
        image = TF.pil_to_tensor(s["image"].convert("RGB")).float() / 255.0
        return image, int(s["image_id"])


def test_collate_fn(batch):
    images, ids = zip(*batch)
    return list(images), list(ids)


test_ds = CPPETestDataset(test_hf)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=test_collate_fn,
)

print("Test images:", len(test_ds))

### Sau cell này

Nếu test dataset code vô tình truy cập annotation để chọn threshold/model, bạn đang phá nguyên tắc private test.

Mọi quyết định hyperparameter phải được khóa từ validation trước bước này.

# 28. Inference test và tạo submission JSON

### Mục tiêu

Chuyển output Faster R-CNN:

- bbox `xyxy` → COCO `xywh`;
- label model `1..5` → category dataset `0..4`;
- giữ confidence score.

Đối với metric AP, không nên filter quá mạnh. Baseline dùng threshold thấp `0.05` từ model.

In [ ]:
@torch.inference_mode()
def create_submission(model, data_loader, device, output_path):
    model.eval()
    predictions = []

    for images, image_ids in tqdm(data_loader, desc="Test inference"):
        images_dev = [img.to(device, non_blocking=True) for img in images]
        outputs = model(images_dev)

        for image_id, out in zip(image_ids, outputs):
            boxes = out["boxes"].detach().cpu()
            scores = out["scores"].detach().cpu()
            labels = out["labels"].detach().cpu()

            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = map(float, box.tolist())
                predictions.append({
                    "image_id": int(image_id),
                    "category_id": int(label) - 1,
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": float(score),
                })

    output_path = Path(output_path)
    output_path.write_text(json.dumps(predictions), encoding="utf-8")
    return predictions

SUBMISSION_PATH = RUN_DIR / "submission.json"
submission = create_submission(best_model, test_loader, DEVICE, SUBMISSION_PATH)

print("# predictions:", len(submission))
print("Saved:", SUBMISSION_PATH)
print("First 3 predictions:", submission[:3])

### Sau cell này — đừng submit ngay

Trước khi nộp, validate format.

Các lỗi contest rất đau nhưng dễ tránh:

- category 1–5 trong khi BTC cần 0–4;
- bbox đang xyxy nhưng ghi là xywh;
- width/height âm;
- score NaN;
- image_id sai;
- file rỗng.

Cell cuối sẽ kiểm tra tự động.

# 29. Validate submission

### Mục tiêu

Đảm bảo submission có cấu trúc hợp lệ trước khi upload.

Validation này không biết “prediction đúng hay sai”; nó chỉ chống lỗi format/kỹ thuật.

In [ ]:
def validate_submission(predictions, test_hf):
    valid_image_ids = {int(test_hf[i]["image_id"]) for i in range(len(test_hf))}
    problems = []

    if len(predictions) == 0:
        problems.append("Submission is empty")

    for i, p in enumerate(predictions):
        if p.get("image_id") not in valid_image_ids:
            problems.append(f"row {i}: invalid image_id")
        if p.get("category_id") not in range(NUM_CLASSES):
            problems.append(f"row {i}: invalid category_id={p.get('category_id')}")

        bbox = p.get("bbox", [])
        if len(bbox) != 4:
            problems.append(f"row {i}: bbox length != 4")
        else:
            x, y, w, h = bbox
            if not all(np.isfinite([x, y, w, h])):
                problems.append(f"row {i}: non-finite bbox")
            if w <= 0 or h <= 0:
                problems.append(f"row {i}: non-positive width/height")

        score = p.get("score")
        if score is None or not np.isfinite(score) or not (0 <= score <= 1):
            problems.append(f"row {i}: invalid score={score}")

        if len(problems) >= 20:
            break

    return problems

submission_problems = validate_submission(submission, test_hf)

if submission_problems:
    print("Submission problems:")
    for x in submission_problems:
        print("-", x)
else:
    print("Submission format sanity: OK")

### Sau cell này

**Nếu OK:** file đủ điều kiện kỹ thuật để submit.

**Nếu lỗi:** sửa format trước, không retrain model. Đây là lỗi pipeline chứ không phải model quality.

Trong contest nên tạo một submission thử **sớm**, ngay sau baseline đầu tiên, để xác minh server chấp nhận format. Đừng để lần submit đầu tiên rơi vào 10 phút cuối.

# 30. Cách tận dụng AI của BTC mà không phụ thuộc vào AI

AI nên được dùng ở chỗ **giảm thời gian suy luận/debug**, không dùng thay thế toàn bộ tư duy pipeline.

## Lúc nên hỏi AI

### Sau EDA

Đưa summary ngắn:

```text
Metric: mAP50
class counts: ...
median relative bbox area từng class: ...
% bbox nhỏ: ...
GPU/time còn lại: ...
Hãy xếp hạng 3 experiment expected gain cao nhất và giải thích ngắn.
```

### Khi mAP bất thường

```text
Faster R-CNN, bbox input xyxy, labels 1..5.
Loss giảm nhưng mAP≈0.
Một target ví dụ: ...
Một output ví dụ: ...
Cho checklist 5 lỗi pipeline có xác suất cao nhất.
```

### Sau baseline

```text
mAP50=...
mAP75=...
AP_small=...
per-class AP=...
Class X recall thấp, precision cao.
Còn 3 giờ.
Đề xuất tối đa 3 thay đổi có thứ tự ưu tiên.
```

## Lúc không nên phí AI

- viết code đếm class đơn giản;
- hỏi lại công thức bbox mà bạn đã xác nhận;
- generate cả notebook mới khi pipeline hiện tại đã chạy;
- hỏi quá rộng như “làm sao tăng điểm?”.

## Quy tắc sử dụng AI contest

1. Prompt phải chứa **metric + hiện tượng + số liệu**.
2. Yêu cầu AI **xếp hạng** thay vì liệt kê 20 ý tưởng.
3. Sau câu trả lời, chỉ chọn experiment có hypothesis rõ.
4. Metric validation mới là bằng chứng cuối cùng.

# 31. Checklist cuối cùng trước khi coi mình “contest-ready”

Bạn nên tự làm được toàn bộ những việc sau mà không cần xem lời giải:

### Data

- [ ] Xác định bbox format đúng.
- [ ] Kiểm tra annotation lỗi.
- [ ] Đếm object/class/image presence.
- [ ] Phân tích relative bbox area.
- [ ] Nhìn ground-truth bằng mắt.

### Validation

- [ ] Tạo split.
- [ ] Kiểm tra class distribution.
- [ ] Khóa split cho mọi experiment.

### Baseline

- [ ] Model hợp lệ với rule pretrained.
- [ ] Smoke test 1 batch.
- [ ] Train với AMP.
- [ ] Save best checkpoint.

### Evaluation

- [ ] mAP50.
- [ ] mAP@[.50:.95].
- [ ] AP75.
- [ ] AP small/medium/large.
- [ ] per-class AP.
- [ ] visualize FP/FN/localization.

### Experiment

- [ ] Mỗi experiment có hypothesis.
- [ ] Chỉ thay một nhóm yếu tố chính.
- [ ] Có experiment log.
- [ ] Không chọn model bằng train loss.

### Submission

- [ ] Freeze model trước test.
- [ ] Convert label/bbox đúng.
- [ ] Check NaN/negative boxes.
- [ ] Test server format sớm.
- [ ] Lưu weights + config + split + history.

---

# Decision tree 30 giây trong phòng thi

```text
mAP thấp?
│
├─ mAP ≈ 0
│  └─ check bbox/label/metric pipeline trước
│
├─ AP_small rất thấp
│  └─ resolution ↑ → anchor nhỏ nếu cần
│
├─ AP50 cao, AP75 thấp
│  └─ localization/resolution/train thêm
│
├─ 1 class chết
│  └─ class count → size → recall/precision → inspect ảnh
│
├─ train tốt, val xấu
│  └─ overfit → augmentation/freeze/regularize
│
└─ cả train và val đều yếu
   └─ underfit → epoch/LR/backbone/model capacity
```

# Tài liệu tham khảo

- CPPE-5 dataset: https://huggingface.co/datasets/rishitdagli/cppe-5
- Torchvision Faster R-CNN ResNet50-FPN: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html
- TorchMetrics MeanAveragePrecision: https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision

Notebook được thiết kế cho **đề luyện mô phỏng**, không phải quy chế chính thức OlpAI 2026. Khi thi thật, rule của từng task luôn ưu tiên cao hơn config trong notebook này.